# 05 Test results
Latest run per model, by training audio (raw vs trimmed). EER, minDCF and F1 recomputed from each run's score
files. F1 uses the dev EER threshold with spoof as the positive class; minDCF uses the project costs (miss 1,
false alarm 10, spoof prior 0.05).

In [ ]:
import pandas as pd
from audio_df.helpers.metrics import compute_eer, min_dcf, prf1_at_threshold
from audio_df.helpers.research.runs import collect, eer

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
SETS = ["dev", "eval", "eval_mp3_32k", "eval_noise_10db", "eval_opus_16k", "eval_resample_8k", "itw",
        "asvspoof2021_eval", "asvspoof2021_hidden", "asvspoof2021df_eval", "asvspoof2021df_hidden"]
results = collect("../runs", MODELS)

def scores_metrics(scores):  # {metric: {testset: value}} for one model
    thr = compute_eer(scores["dev"].score, scores["dev"].label, higher_is_spoof=False)[1]
    out = {"EER %": {}, "minDCF": {}, "F1 (spoof)": {}}
    for t in SETS:
        if t in scores:
            s = scores[t]
            out["EER %"][t] = eer(s)
            out["minDCF"][t] = min_dcf(s.score, s.label, higher_is_spoof=False)[0]
            out["F1 (spoof)"][t] = prf1_at_threshold(s.score, s.label, thr, higher_is_spoof=False)["f1"]
    return out

per_model = {k: scores_metrics(v["scores"]) for k, v in results.items()}

def metric_table(name):  # mean over the 10 test sets, only when a model has all of them
    t = pd.DataFrame({k: m[name] for k, m in per_model.items()}).T.reindex(columns=SETS).sort_index()
    t.index.names = ["model", "train"]
    t["mean"] = t[SETS[1:]].mean(axis=1, skipna=False)
    return t

pd.DataFrame({k: {"artifact": v["artifact"], "runs": ", ".join(v["runs"])} for k, v in results.items()}).T

## EER % per test set
D2/D3 are labelled with their WavLM layer (L12 = round 2, L6 = round 3). A1/A4 DF results are on 10,000-clip
samples (notebook 01).

In [ ]:
metric_table("EER %").round(2)

## minDCF per test set
Lower is better; 1.0 is no better than always answering the cheaper class.

In [ ]:
metric_table("minDCF").round(3)

## F1 per test set
At the dev EER threshold, spoof = positive, higher is better. Spoof is about 90% of most sets, so F1 stays high
even for weak models; read it next to EER and minDCF.

In [ ]:
metric_table("F1 (spoof)").round(3)

## D2 vs D3 (round 3, WavLM layer 6)

In [ ]:
pairs = {m: metric_table(m).xs("trimmed", level="train").loc[["d2 L6", "d3 L6"]] for m in ["EER %", "minDCF"]}
cmp = pd.DataFrame({f"{m} {k}": pairs[m].loc[f"{k} L6"] for m in pairs for k in ["d2", "d3"]})
cmp["better (EER)"] = (cmp["EER % d2"] < cmp["EER % d3"]).map({True: "D2", False: "D3"})
cmp.round(3)

D2 wins 7 of 11 sets and the mean (EER 11.0% vs 11.5%, minDCF 0.264 vs 0.284). D3 only wins on audio derived from the 2019 corpus (2019 eval, MUSAN noise, Opus, 8 kHz); D2 wins every cross-dataset set (In-the-Wild 19.9% vs 22.7%, 2021 LA hidden, 2021 DF eval and hidden). D3 fits the training corpus slightly better; D2 generalises better. Both from one training run each.

## Per attack, 2019 eval (EER %)

In [ ]:
def per_attack(df):
    bona = df[df.label == "bonafide"]
    return {a: eer(pd.concat([bona, g])) for a, g in df[df.label == "spoof"].groupby("attack_id")}

pd.DataFrame({k: per_attack(v["scores"]["eval"]) for k, v in results.items()}).T.round(1)

## Conclusion
- Trimmed training fixed the silence shortcut for the WavLM models; the LFCC-only models (A1, A4) stay weak on every metric (A4's minDCF is close to 1).
- Round 3 D2/D3 (layer 6) beat A2 on mean EER and minDCF over the 10 test sets (minDCF D2 0.264, D3 0.284, A2 0.303); F1 is about the same for all three (≈0.92) because spoof dominates every set.
- D2 is the best general model, D3 the best on 2019-like audio. In-the-Wild stays around 20% for every model (notebooks 04, 07).